In [30]:
from pathlib import Path
import os
import webdataset as wds


def parse_directory(writer, root_path, split="test", origin="artificial"):
    files = os.listdir(root_path)
    for fidx, file_key in enumerate(files):
        if file_key.startswith("."): continue
        item_path = root_path / file_key

        print(f"\r{split} {fidx}/{len(files)} {file_key}" + " " * 20, end="")

        with open(item_path / "controls.csv", "r") as csvfile:
            controls_csv = csvfile.read()
        with open(item_path / "vehicle.csv", "r") as csvfile:
            vehicle = csvfile.read()
        with open(item_path / "optimal_path.csv", "r") as csvfile:
            optimal_path = csvfile.read()
        with open(item_path / "boundaries.csv", "r") as csvfile:
            boundaries = csvfile.read()
        with open(item_path / "track.csv", "r") as csvfile:
            track = csvfile.read()
        with open(item_path / "slip.csv", "r") as csvfile:
            slip = csvfile.read()
        with open(item_path / "accelerations.csv", "r") as csvfile:
            accelerations = csvfile.read()
        with open(item_path / "tire_forces.csv", "r") as csvfile:
            tire_forces = csvfile.read()
        with open(item_path / "states.csv", "r") as csvfile:
            states = csvfile.read()
        with open(item_path / "states.csv", "r") as csvfile:
            states = csvfile.read()
        with open(item_path / "times.txt", "r") as csvfile:
            times = csvfile.read().split("\n")

        compute_time = float(times[0].split(": ")[1])
        laptime_time = float(times[1].split(": ")[1])

        data = {
            "__key__": f"{split}/{file_key.replace(".csv", "")}",
            "id": file_key.replace(".csv", ""),
            "origin": origin,
            "controls": controls_csv,
            "vehicle": vehicle,
            "optimal_path": optimal_path,
            "boundaries": boundaries,
            "track": track,
            "slip": slip,
            "accelerations": accelerations,
            "tire_forces": tire_forces,
            "states": states,
            "metadata-compute-time": str(compute_time),
            "metadata-laptime": str(laptime_time),
        }

        if " - " in file_key:
            vehicle_name, track_name, track_variant = file_key.replace(".csv", "").split(" - ")
            data["metadata-vehicle"] = vehicle_name
            data["metadata-track"] = track_name
            data["metadata-variant"] = track_variant

        writer.write(data)


In [31]:
_ROOT = Path("/Users/belle/Developer/MlLapSim/dataset/")

# with wds.ShardWriter(str(_ROOT / "lapsim-train-%02d.tar"), maxcount=10000) as writer:
#     parse_directory(
#         writer,
#         Path('/Users/belle/Downloads/ArtificialTraining'),
#         split="train",
#         origin="artificial"
#     )
#
# with wds.ShardWriter(str(_ROOT / "lapsim-validation-%01d.tar"), maxcount=10000) as writer:
#     parse_directory(
#         writer,
#         Path('/Users/belle/Downloads/Artificial Validation'),
#         split="validation",
#         origin="artificial"
#     )
#
# with wds.ShardWriter(str(_ROOT / "lapsim-test-%01d.tar"), maxcount=10000) as writer:
#     parse_directory(
#         writer,
#         Path('/Users/belle/Downloads/Artificial Test'),
#         split="test",
#         origin="artificial"
#     )

with wds.ShardWriter(str(_ROOT / "lapsim-real-%01d.tar"), maxcount=10000) as writer:
    parse_directory(
        writer,
        Path('/Users/belle/Downloads/Real Test'),
        split="test",
        origin="real"
    )


# writing /Users/belle/Developer/MlLapSim/dataset/lapsim-real-0.tar 0 0.0 GB 0
test 1809/4579 Generic Lola T70 Mk3B - UK Brands Hatch - Indy Circuit.csv                    # writing /Users/belle/Developer/MlLapSim/dataset/lapsim-real-1.tar 1809 3.0 GB 1809
test 3581/4579 Mazda Mx5 Mk3 - US Daytona - Road Course.csv                     Internacional + T9 Chicane.csv                                                    # writing /Users/belle/Developer/MlLapSim/dataset/lapsim-real-2.tar 1772 3.0 GB 3581
test 4578/4579 Mazda Mx5 Mk3 - US Las Vegas Motor Speedway - Outfield North Road Course.csv                                      v                                   

In [33]:

dataset = wds.WebDataset("/Users/belle/Developer/MlLapSim/dataset/lapsim-real-{0..2}.tar")

print(dataset.length)

# print(dir(dataset))


for item in dataset:
    print(list(item))
    vehicle = item["vehicle"].decode().split("\n")[1:]
    vehicle_params = [x.split(",") for x in vehicle if "," in x]

    print({datum[0]: float(datum[1]) for datum in vehicle_params})
    break


-1
['__key__', '__url__', 'accelerations', '__local_path__', 'boundaries', 'controls', 'id', 'metadata-compute-time', 'metadata-laptime', 'metadata-track', 'metadata-variant', 'metadata-vehicle', 'optimal_path', 'origin', 'slip', 'states', 'tire_forces', 'track', 'vehicle']
{'width': 1.7, 'track_front': 1.7, 'track_rear': 1.7, 'wheel_base_front': 1.0865, 'wheel_base_rear': 1.5635, 'mass': 1265.0, 'k_drive_front': 1.0, 'k_roll': 0.55, 'tyre_friction': 1.0, 'max_power': 260000.0, 'cog_height': 0.4, 'f_drive_max': 49638.600000000006, 'lift_coeff_front': 0.164, 'lift_coeff_rear': 0.236, 'v_max': 69.44444444, 'drag_coeff': 0.65, 'yaw_inertia': 2263.476, 'gamma_y': 3373.8029048075537, 'f_z0': 3102.4125000000004, 'k_brake_front': 0.7409433962, 'f_brake_max': 14338.6623454321}


/Users/belle/Developer/MlLapSim/venv/lib/python3.14/site-packages/webdataset/compat.py:379: UserWarning: WebDataset(shardshuffle=...) is None; set explicitly to False or a number
  warnings.warn("WebDataset(shardshuffle=...) is None; set explicitly to False or a number")


['__abstractmethods__',
 '__add__',
 '__annotate_func__',
 '__annotations_cache__',
 '__class__',
 '__class_getitem__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__enter__',
 '__eq__',
 '__exit__',
 '__firstlineno__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getitem__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__iter__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__orig_bases__',
 '__parameters__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__slotnames__',
 '__slots__',
 '__static_attributes__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_abc_impl',
 'append',
 'batched',
 'close',
 'compose',
 'create_url_iterator',
 'decode',
 'extract_keys',
 'invoke',
 'iterator',
 'iterator1',
 'length',
 'listed',
 'lmdb_cached',
 'log_keys',
 'map',
 'map_dict',
 'map_tuple',
 'mcached',
 'nsamples',
 'pipeline',
 'rename',
 'rename_keys',
 'repeat',
 'repetitions',
 'rsampl